In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

DATA_SNIPPET = """Time,Temperature,Pressure,Wind Speed
12:00,17.2,1012,2.1
12:30,17.5,1011,2.5
13:00,16.8,991,6.9
13:30,16.5,1010,2.8
14:00,16.4,1011,2.6
"""

PROMPT_TEMPLATE = f"""
Context: A junior researcher and a senior scientist are looking at the attached 2017 data.

Data:
```csv
{DATA_SNIPPET.strip()}
```

Dialogue:
Junior: 'The 13:00 reading shows a Temperature of 16.8, but the Pressure dropped to 991 suddenly.'
Senior: 'Right... and I suppose the Wind Speed of 6.9 is just a "coincidence" given the sensor was shaking?'

Question: Based on the Senior's response and the data provided, does the Senior believe the 13:00 reading is accurate? What is the Senior implying about the equipment? Use the data to support your interpretation of his tone.
"""

@kbench.task(name="single_theory_of_mind_check", store_task=False)
def single_theory_of_mind_check(llm, prompt_text: str) -> bool:
    response = llm.prompt(prompt_text)

    assessment = kbench.assertions.assess_response_with_judge(
        criteria=[
            "Correctly identifies that the Senior believes the 13:00 reading is inaccurate.",
            "Correctly infers that the Senior is implying the equipment is malfunctioning or faulty.",
            "Uses the provided data (specifically the sudden drop in pressure and high wind speed) to support the interpretation.",
            "Accurately interprets the Senior's tone as sarcastic or skeptical.",
        ],
        response_text=response,
        judge_llm=kbench.judge_llm
    )

    if assessment is None:
        kbench.assertions.assert_fail(expectation="Judge LLM failed to return a valid assessment.")
        return False

    all_passed = all(result.passed for result in assessment.results)
    if not all_passed:
        failed_reasons = [f"Criterion '{r.criterion}' failed: {r.reason}" for r in assessment.results if not r.passed]
        kbench.assertions.assert_fail(expectation="; ".join(failed_reasons))

    return all_passed

@kbench.task(name="social_cognition_test")
def social_cognition_test(llm) -> tuple[int, int]:
    evaluation_data = pd.DataFrame({"prompt_text": [PROMPT_TEMPLATE] * 10})

    with kbench.client.enable_cache():
        runs = single_theory_of_mind_check.evaluate(
            llm=[llm],
            evaluation_data=evaluation_data,
            n_jobs=2,
            remove_run_files=True
        )

    eval_df = runs.as_dataframe()
    successes = int(eval_df.result.sum())
    total = len(eval_df)

    return successes, total

social_cognition_test.run(kbench.llm)